# Programación funcional: pereza y orden superior

Este notebook explica:

- Evaluación perezosa y generadores.
- Funciones lambda.
- Funciones de orden superior.
- `filter`, `map` y `reduce`.
- Generadores de números primos.
- Decoradores y memoización.


## Evaluación perezosa

Una expresión se evalúa solamente cuando es necesaria.  Está en casi todos los lenguajes de programación.


In [1]:
# No da error
l = [1, 3, 5, 7, 2, 4, 5]
val = 0
i = 0
while i < len (l ) and l[i]!= val :
    i += 1

In [2]:
# Se produce un error
i = 0
while l[i]!= val and i < len (l):
    i += 1

IndexError: list index out of range

## Crear generadores con yield

Python tiene constructores perezosos. Los generadores permiten trabajar con secuencias potencialmente infinitas sin almacenarlas completas en memoria.

Con `yield` podemos crear generadores. Funciona de forma similar a `return`, pero en lugar de devolver un valor y terminar la ejecución de la función, `yield` pausa la función, devuelve un valor y guarda todo el estado interno para continuar en el mismo punto la próxima vez que se le solicite un dato.

Puedo hacer una lista con todos los números enteros:


In [3]:
from collections.abc import Iterator, Callable, Iterable
def all_natural_numbers() -> Iterator[int]:
    n = 0
    while True:
        yield n
        n += 1

numbers = all_natural_numbers()
[next(numbers) for _ in range(10)]


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

* `all_natural_numbers` no construye una lista infinita.  
* Cada llamada a `next` calcula únicamente el siguiente valor.


## Funciones lambda 



Las **funciones anónimas** son aquellas que no tienen nombre y se refieren a una única instrucción. Se declaran con la palabra reservada `lambda`.

Son funciones cortas. Están sintácticamente restringidas a una sola expresión.

Las funciones Lambda pueden ser usadas en cualquier lugar donde sea requerido un objeto de tipo función.  Semánticamente, son solo
azúcar sintáctica. Al igual que las funciones anidadas, las funciones `lambda` pueden hacer referencia a variables desde el ámbito que las contiene.

In [4]:
# Ejemplo
# función normal
def producto(a):  
    return a * 2

# la función anónima equivalente:
resultado = lambda x: x * 2

In [ ]:
# ejemplo 
add = lambda x,y: x+y

def add (x , y ):
    return x+y

In [5]:
# ejemplo 

def is_even(x ):
    return x %2==0

is_even = lambda x: x % 2 == 0


In [6]:
[is_even(x) for x in range(10)]



[True, False, True, False, True, False, True, False, True, False]

## Orden superior y pereza

Las funciones en python son objetos de primera clase.  Esto significa que las funciones pueden tratarse como cualquier otra variable: se pueden asignar a variables, pasar como argumentos y devolver desde otras funciones.

In [7]:
def is_prime ( num : int ) -> bool :
    """
    Finds out is num is a prime number
    """
    if num<2:
        return False
    i = 2
    while i*i<=num and num%i!=0:
        i += 1
    return i*i > num

In [ ]:
def myfilter( prop : Callable[[int], bool ],
                 lst : Iterable[int]) -> Iterable[int]:

    """
    >>> list ( myfilter( lambda x : x %2==0 , range (2 ,10)))
    [2 , 4 , 6 , 8]
    >>> list ( myfilter( is_prime , range (2 , 10)))
    [2 , 3 , 5 , 7]
    """
    rlst = []
    for a in lst :
        if prop(a):
            yield a     

In [ ]:
list(myfilter( lambda x : x %2==0 , range (2 ,10)))

In [ ]:
list(myfilter( is_prime , range (2 , 10)))

##  `map`

La función `map()` es una función nativa de orden superior que aplica una función específica a cada uno de los elementos de un objeto iterable (como listas, tuplas o conjuntos) y devuelve un objeto generador/iterador con los resultados.
`map` aplica una función a cada elemento y también es perezoso.


In [8]:
# simulación de la función map
def mymap(fun: Callable[[int], int],
          iterable: Iterable[int]) -> Iterator[int]:
    for element in iterable:
        yield fun(element)

list(mymap(lambda x: 2 * x, range(2, 10)))


[4, 6, 8, 10, 12, 14, 16, 18]

In [ ]:
# map nativa
list(map(lambda x: 2 * x, range(2, 10)))

In [ ]:
def next_prime(num: int) -> int:
    candidate = num + 1
    while not is_prime(candidate):
        candidate += 1
    return candidate

list(mymap(next_prime, [2, 5, 6, 8, 11]))


##  `filter`

La función `filter()` es otra función nativa de orden superior en Python. Sirve para filtrar los elementos de un iterable manteniendo únicamente aquellos para los cuales una función condicional devuelve True.

A diferencia de `map()` (que modifica todos los elementos), filter() selecciona o descarta elementos sin alterarlos.

`filter` es perezoso: devuelve un iterador y solo evalúa elementos cuando se solicitan.


In [ ]:
list(filter(is_prime, range(2, 30)))

__Generador perezoso de primos__

Se aplica una versión perezosa de la criba de Eratóstenes.


In [ ]:
def div(p:int) -> Callable[[int], bool]:
    return lambda x: x % p != 0


def gen_primes() -> Iterator[int]:
    candidates = filter(lambda x: x > 1, all_natural_numbers())

    while True:
        prime = next(candidates)
        yield prime
        candidates = filter(div(prime), candidates)

primes = gen_primes()
[next(primes) for _ in range(15)]


## `reduce`

`reduce` sirve para acumular o colapsar los elementos de un iterable en un único valor final (por ejemplo: sumar todos los elementos, hallar el producto total o encontrar el valor máximo).. A diferencia de `map()` y `filter()`, reduce() no es una función nativa global en Python 3; se encuentra dentro del módulo **functools**.



In [9]:
from functools import reduce
reduce(lambda x, y: max(x, y), [34, 45, 55, 29], 0)

55

In [10]:

def myreduce(fun: Callable[[T, T], T],
             iterable: Iterable[T],
             initial: T) -> T:
    result = initial
    for element in iterable:
        result = fun(result, element)
    return result

myreduce(lambda x, y: max(x, y), [34, 45, 55, 29], 0)


55

In [ ]:
myreduce(lambda x, y: max(x, y), [34, 45, 55, 29], 0)


Para paralelizar correctamente un `reduce`, la operación suele ser:

- Asociativa: $f(a, f(b, c)) = f(f(a, b), c)$
- Preferiblemente conmutativa: $f(a, b) = f(b, a)$
- Definida a partir de un elemento neutro: Existe $e$ tal que $f(x, e) = x$. <br>*(Ej: $0$ en suma, $1$ en producto).*


## Decoradores

Un **decorador** en Python es una función que toma otra función como argumento, añade alguna funcionalidad extra y devuelve una nueva función modificada. Todo esto se hace sin alterar el código fuente de la función original. Se aplican utilizando la sintaxis de la "arroba" (`@nombre_decorador`).


In [11]:
def debug(function):
    def wrapper(*args, **kwargs):
        print(f"Llamando a {function.__name__}")
        result = function(*args, **kwargs)
        print(f"Resultado: {result}")
        return result

    return wrapper

@debug
def square(number):
    return number * number

square(5)


Llamando a square
Resultado: 25


25

__Ejemplo:__

In [12]:
from datetime import datetime, timedelta

def debug(func):
    def wrapper(*args, **kwargs):
        print(f'Calling {func.__name__} with {args} and {kwargs}')
        ini = datetime.now()
        val = func(*args, **kwargs)
        end = datetime.now()
        unit = timedelta(milliseconds=1)
        print(f'{func.__name__} returned {val}, took {(end-ini)/unit} ms.')
    return wrapper

@debug
def is_prime(n: int) -> bool:
    if n < 0:
        return is_prime(-n)
    if n < 2:
        return False
    i = 2
    while i * i <= n and n % i != 0:
        i += 1
    return i * i > n

is_prime(2**100+1)

Calling is_prime with (1267650600228229401496703205377,) and {}
is_prime returned False, took 0.041 ms.


__Ejemplo:__

Este código implementa un **decorador de memoización** (caché) para optimizar la función recursiva de Fibonacci. Permite transformar un algoritmo de complejidad exponencial $\mathcal{O}(2^n)$ en uno de complejidad lineal $\mathcal{O}(n)$.

In [13]:
def cached(function):
    cache = {}

    def wrapper(*args):
        if args not in cache:
            cache[args] = function(*args)
        return cache[args]

    return wrapper

@cached
def fibonacci_1(n):
    if n < 2:
        return n
    return fibonacci_1(n - 1) + fibonacci_1(n - 2)

def fibonacci_2(n):
    if n < 2:
        return n
    return fibonacci_2(n - 1) + fibonacci_2(n - 2)


In [14]:
ini = datetime.now()
print(fibonacci_1(30))
end = datetime.now()
unit = timedelta(milliseconds=1)
print(f'fibonacci_1 took {(end-ini)/unit} ms.')

ini = datetime.now()
print(fibonacci_2(30))
end = datetime.now()
print(f'fibonacci_2 took {(end-ini)/unit} ms.')


832040
fibonacci_1 took 0.598 ms.
832040
fibonacci_2 took 79.313 ms.


* **cache = {}** (Closure / Cierre):
Es un diccionario local definido dentro de cached. Gracias al concepto de closure, la función interna wrapper conserva el acceso a este diccionario entre diferentes llamadas, manteniendo los resultados guardados en memoria RAM.

## Referencias

* Filter [https://docs.python.org/3/builtins/functions.html#filter](https://docs.python.org/3/builtins/functions.html#filter)